# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Print all record sets and their fields by @id
print("Record sets available in this dataset:\n")
record_sets = metadata.record_sets
for record_set in record_sets:
    print(f"- Record Set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'N/A')}")
    fields = record_set.get('field', [])
    # makesure fields is always a list
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields and their @ids:")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name', 'N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print(f"Record set @ids: {record_set_ids}")

# For demonstration, just extract first available record set
if record_set_ids:
    selected_record_set = record_set_ids[0]
    print(f"Extracting records from: {selected_record_set}\n")
    records = list(dataset.records(record_set=selected_record_set))
    df = pd.DataFrame(records)
    dataframes[selected_record_set] = df
    print("Columns extracted:")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
#---
# We try to pick a numeric field (column) for demonstration. List numeric columns, or guess from column names or types.
import numpy as np

target_df = None
target_record_set_id = None
if dataframes:
    # Pick the dataframe and list numeric-like columns
    target_record_set_id = list(dataframes.keys())[0]
    target_df = dataframes[target_record_set_id]
    print(f"Exploring DataFrame from Record Set: {target_record_set_id}")

    numeric_candidates = [col for col in target_df.columns if any(x in col.lower() for x in ['count', 'mw', 'mass', 'weight', 'abund', 'peptide', 'quant'] )]
    print(f"Possible numeric fields: {numeric_candidates}")
    if not numeric_candidates:
        # Fallback to numeric columns by dtype
        numeric_candidates = target_df.select_dtypes(include=[np.number]).columns.tolist()
        print(f"Numeric columns by dtype: {numeric_candidates}")

    if numeric_candidates:
        # For demo, analyze the first candidate
        numeric_field = numeric_candidates[0]
        print(f"\nAnalyzing numeric field: {numeric_field}\n")

        # Try coercing field to float (if not already)
        target_df[numeric_field] = pd.to_numeric(target_df[numeric_field], errors='coerce')

        # Choose arbitrary threshold (e.g. 10)
        threshold = 10
        filtered_df = target_df[target_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical/likely useful field (e.g. name/description etc.)
        # Pick candidate group field (but not the analyzed numeric)
        group_candidates = [col for col in target_df.columns if col != numeric_field and target_df[col].dtype == object]
        group_field = group_candidates[0] if group_candidates else None
        if group_field is not None:
            print(f"Grouping on field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No suitable categorical/grouping field found.")
    else:
        print("No numeric fields found.")
else:
    print("No dataframe found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Distribution/Histogram of the numeric field
import matplotlib.pyplot as plt

if target_df is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    target_df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()

    # If group_field exists, boxplot by group
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,5))
        target_df.boxplot(column=numeric_field, by=group_field, grid=False, rot=90)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print('No numeric field or DataFrame found for plotting.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- 
Add your summary here. Example: 
- The dataset provides detailed mass spectrometry data including protein counts and abundances.
- After filtering and normalization, key measurements can be compared across conditions.
- Visualizations reveal distribution skews, outliers, and grouping characteristics.
-->
